# LeetCode #1129 — Shortest Path with Alternating Colors

**Medium · Python 3 · Graph / BFS / Queue / State Expansion**  
[Đề gốc LeetCode](https://leetcode.com/problems/shortest-path-with-alternating-colors/) · [Đối chiếu đề và giới hạn](https://leetcode.doocs.org/en/lc/1129/)

```python
class Solution:
    def shortestAlternatingPaths(
        self, n: int, redEdges: List[List[int]], blueEdges: List[List[int]]
    ) -> List[int]:
        ...
```

**Mục tiêu:** từ bài toán đường đi ngắn nhất trên đồ thị không trọng số, học cách bổ sung **màu cạnh cuối** vào trạng thái BFS. Mọi kiến thức `list`, graph, queue, set đều được giới thiệu trước.

## 1. Bạn sẽ học được gì?

1. Đọc đúng đồ thị **có hướng**; hiểu vòng tự khép (*self-loop*) và cạnh song song (*parallel edges*).
2. Biểu diễn cạnh đỏ/xanh bằng **adjacency list** (danh sách kề).
3. Phân biệt BFS theo **nút** với BFS theo **trạng thái `(nút, màu cạnh cuối)`**.
4. Giải thích vì sao BFS tìm được đường đi ngắn nhất khi mỗi cạnh có độ dài 1.
5. Tránh lỗi đánh dấu visited quá sớm hoặc đánh dấu thiếu màu.
6. Viết hàm nộp và kiểm thử độc lập bằng cách duyệt lớp/relaxation.

## 2. Đề bài, ví dụ và constraints

Cho `n` đỉnh số `0...n-1`. Cạnh từ `u` sang `v` có thể **đỏ** (`redEdges`) hoặc **xanh** (`blueEdges`); đồ thị **có hướng**, có thể có cạnh trùng và cạnh `u→u`.

Bắt đầu ở đỉnh `0`. Đường đi hợp lệ phải **đổi màu sau mỗi cạnh**: đỏ→xanh→đỏ..., hoặc xanh→đỏ→xanh... Trả `answer[x]` = **số cạnh ít nhất** đi từ 0 tới x; nếu không thể đi thì `-1`. Riêng `answer[0]=0` vì đường đi không có cạnh là hợp lệ.

- Ví dụ 1: `n=3`, đỏ `[[0,1],[1,2]]`, xanh `[]` → `[0,1,-1]` (hai cạnh đỏ liên tiếp không hợp lệ).
- Ví dụ 2: `n=3`, đỏ `[[0,1]]`, xanh `[[2,1]]` → `[0,1,-1]` (không có đường đi từ 0 tới 2).
- Ví dụ minh họa thêm: đỏ `[[0,1]]`, xanh `[[1,2]]` → `[0,1,2]`.

**Constraints:** `1 <= n <= 100`; số cạnh mỗi màu `0..400`; mỗi cạnh có 2 đỉnh và các chỉ số thuộc `[0,n-1]`.

## 3. Nền tảng: Đồ thị có hướng là gì?

Một cạnh `0→1` chỉ đi **từ 0 sang 1**, không mặc nhiên đi ngược 1→0. Với:

- cạnh đỏ `0→1`,
- cạnh xanh `1→2`,

đường đi `0→1→2` có 2 cạnh và đổi màu đúng. Ngược lại `2→1→0` **không tồn tại** nếu đề không cung cấp cạnh ngược.

`redEdges` và `blueEdges` là các **danh sách cặp** `[u,v]`, không phải ma trận kề.

In [1]:
red = [[0,1]]
blue = [[1,2]]
for u, v in red:
    print('ĐỎ: ',u,'->',v)
for u, v in blue:
    print('XANH:',u,'->',v)
assert [0,1] in red
assert [1,0] not in red

ĐỎ:  0 -> 1
XANH: 1 -> 2


## 4. Python: list, chỉ số và danh sách kề

`adj[u]` là **danh sách các đỉnh v mà có cạnh u→v**. Trong bài hai màu, cần `adj[color][u]`, quy ước `RED=0`, `BLUE=1`.

`[[[] for _ in range(n)] for _ in range(2)]` tạo **hai nhóm**, mỗi nhóm có n **list độc lập**. Đừng dùng `[[[]]*n]*2`: các list con có thể dùng chung tham chiếu, thêm một cạnh lại làm nhiều ô cùng thay đổi.

In [2]:
RED, BLUE = 0, 1
n = 3
adj_example = [[[] for _ in range(n)] for _ in range(2)]
adj_example[RED][0].append(1)
adj_example[BLUE][1].append(2)
print('Cạnh đỏ từ 0:',adj_example[RED][0])
print('Cạnh xanh từ 1:',adj_example[BLUE][1])
print('Cạnh đỏ từ 1:',adj_example[RED][1])
assert adj_example[RED][0] == [1]
assert adj_example[RED][1] == []

Cạnh đỏ từ 0: [1]
Cạnh xanh từ 1: [2]
Cạnh đỏ từ 1: []


## 5. Python: queue `deque` và BFS (duyệt theo chiều rộng)

BFS lấy trạng thái **vào trước ra trước (FIFO)**. `collections.deque` hỗ trợ:

- `append(x)`: thêm vào cuối;
- `popleft()`: lấy đầu hàng đợi, thời gian `O(1)`;
- `while q:`: còn phần tử để xử lý.

Nếu mỗi cạnh dài 1, BFS xét những đường đi dài 0, rồi 1, rồi 2... Do đó **lần đầu tới trạng thái** là đường ngắn nhất tới trạng thái đó.

In [3]:
from collections import deque
q = deque(['lượt 0'])
q.append('lượt 1')
q.append('lượt 2')
print(q.popleft(),q.popleft(),q.popleft())
assert not q

lượt 0 lượt 1 lượt 2


## 6. BFS bình thường (chưa xét màu)

Trong đồ thị chỉ có cạnh 0→1→2, BFS sẽ gán khoảng cách `0,1,2`. Đây là **baseline để học BFS**, nhưng **chưa thể dùng nộp bài**, vì bản thường không biết hai cạnh có cùng màu hay không.

In [4]:
def shortest_paths_ignore_colors(n, red_edges, blue_edges):
    g = [[] for _ in range(n)]
    for u,v in red_edges + blue_edges:
        g[u].append(v)
    dist = [-1]*n
    dist[0] = 0
    q = deque([0])
    while q:
        u = q.popleft()
        for v in g[u]:
            if dist[v] == -1:
                dist[v] = dist[u]+1
                q.append(v)
    return dist

print('BFS bỏ qua màu:',shortest_paths_ignore_colors(3, [[0,1],[1,2]], []))
assert shortest_paths_ignore_colors(3, [[0,1],[1,2]], []) == [0,1,2]

BFS bỏ qua màu: [0, 1, 2]


## 7. Tại sao BFS chỉ nhớ `node` sẽ sai?

Giả sử:

- đỏ `0→1`, đỏ `1→2`;
- xanh `0→1`.

Muốn tới 2, ta **phải** đi `0 -xanh→ 1 -đỏ→ 2`. Nếu ta chỉ lưu `visited={1}`, rồi duyệt 0→1 **đỏ trước**, sẽ bỏ qua lần đến 1 **xanh**, mất đường đi hợp lệ.

Đến **cùng một đỉnh**, nhưng **màu cạnh cuối khác** nghĩa là những lựa chọn bước tiếp theo khác nhau. Vì vậy đây là **hai trạng thái riêng**.

In [5]:
red_demo = [[0,1],[1,2]]
blue_demo = [[0,1]]
print('Bỏ màu:',shortest_paths_ignore_colors(3,red_demo,blue_demo))
print('Phải xét màu: 0 -xanh-> 1 -đỏ-> 2, dài 2')
assert shortest_paths_ignore_colors(3,red_demo,blue_demo) == [0,1,2]

Bỏ màu: [0, 1, 2]
Phải xét màu: 0 -xanh-> 1 -đỏ-> 2, dài 2


## 8. Bước đột phá — mở rộng trạng thái

Định nghĩa **trạng thái**: `(u, last_color)` gồm:

- `u`: đỉnh hiện tại;
- `last_color`: màu của **cạnh vừa đi** để đến `u`.

Nếu `last_color=RED`, cạnh tiếp theo chỉ được là BLUE; nếu `last_color=BLUE`, cạnh tiếp theo chỉ được là RED.

**Tại đỉnh xuất phát**, chưa đi cạnh nào, đặt `last_color=-1` (*chưa có màu*). Ta được chọn cả đỏ lẫn xanh làm cạnh đầu.

Một đường đi có thể **lặp lại đỉnh**, vì vậy set `visited` lưu **trạng thái `(u, color)`**, không chỉ lưu `u`.

## 9. Python: `set` để đánh dấu trạng thái

`visited = {(0,-1)}` là set chứa một **tuple**. `tuple` như `(u,c)` là cặp dữ liệu cố định có thể làm phần tử set.

Trong Python, kiểm tra `state in visited` có thời gian trung bình `O(1)`.

**Khi nào đánh dấu?** Ngay lúc **đưa trạng thái vào hàng đợi** để không đẩy trùng nhiều lần. Một trạng thái tối đa được đưa vào BFS một lần.

In [6]:
visited = {(0,-1)}
visited.add((1, RED))
visited.add((1, BLUE))
print('Các trạng thái đã thăm:', sorted(visited))
assert (1,RED) in visited and (1,BLUE) in visited
assert len(visited) == 3

Các trạng thái đã thăm: [(0, -1), (1, 0), (1, 1)]


## 10. Từng bước xây dựng thuật toán chính

1. Tạo `adj[color][u]` chứa các đỉnh v đi được bằng màu `color`.
2. Cho `q=deque([(0,-1,0)])`: `(node, last_color, dist)`.
3. Lấy trạng thái đầu. Nếu `answer[node]` chưa được gán, đặt bằng `dist` (BFS đảm bảo ngắn nhất).
4. Chỉ duyệt các cạnh có màu **khác `last_color`**.
5. Chỉ thêm `(v,new_color)` vào queue nếu chưa thăm cặp này.

Bộ nhớ chỉ cần tối đa `2n+1` trạng thái, dù đồ thị có chu trình.

In [7]:
def shortest_alternating_bfs(n, red_edges, blue_edges):
    g = [[[] for _ in range(n)] for _ in range(2)]
    for u, v in red_edges:
        g[RED][u].append(v)
    for u, v in blue_edges:
        g[BLUE][u].append(v)

    ans = [-1] * n
    q = deque([(0,-1,0)])
    visited = {(0,-1)}
    while q:
        u, last_color, distance = q.popleft()
        if ans[u] == -1:
            ans[u] = distance
        for color in (RED, BLUE):
            if color == last_color:
                continue
            for v in g[color][u]:
                state = (v, color)
                if state not in visited:
                    visited.add(state)
                    q.append((v, color, distance + 1))
    return ans

assert shortest_alternating_bfs(3, [[0,1],[1,2]], []) == [0,1,-1]
assert shortest_alternating_bfs(3, [[0,1]], [[1,2]]) == [0,1,2]
assert shortest_alternating_bfs(3, red_demo, blue_demo) == [0,1,2]

## 11. Trace bằng tay — đỉnh, màu cuối và khoảng cách

Ví dụ `redEdges=[[0,1],[1,2]]`, `blueEdges=[[0,1]]`.

| Bước lấy queue | Trạng thái `(u,màu cuối,d)` | Hành động | answer |
|---:|---|---|---|
| 1 | `(0,không có,0)` | thêm `(1,đỏ,1)` và `(1,xanh,1)` | `[0,-1,-1]` |
| 2 | `(1,đỏ,1)` | không thể dùng cạnh đỏ `1→2` | `[0,1,-1]` |
| 3 | `(1,xanh,1)` | dùng đỏ `1→2`, thêm `(2,đỏ,2)` | `[0,1,-1]` |
| 4 | `(2,đỏ,2)` | ghi đáp án 2 | `[0,1,2]` |

Nếu chỉ visited theo đỉnh, bước 3 sẽ không tồn tại và kết quả bị sai.

In [8]:
def trace_alternating_bfs(n, red_edges, blue_edges):
    g = [[[] for _ in range(n)] for _ in range(2)]
    for u,v in red_edges:g[RED][u].append(v)
    for u,v in blue_edges:g[BLUE][u].append(v)
    q, visited = deque([(0,-1,0)]), {(0,-1)}
    ans = [-1]*n
    colors = {-1:'không có', RED:'đỏ', BLUE:'xanh'}
    while q:
        u, last, d = q.popleft()
        if ans[u] == -1:ans[u] = d
        print('Lấy:',(u,colors[last],d),'| ans:',ans)
        for color in (RED, BLUE):
            if color == last:continue
            for v in g[color][u]:
                if (v,color) not in visited:
                    visited.add((v,color))
                    q.append((v,color,d+1))
                    print('  thêm:',(v,colors[color],d+1))
    return ans

assert trace_alternating_bfs(3,red_demo,blue_demo) == [0,1,2]

Lấy: (0, 'không có', 0) | ans: [0, -1, -1]
  thêm: (1, 'đỏ', 1)
  thêm: (1, 'xanh', 1)
Lấy: (1, 'đỏ', 1) | ans: [0, 1, -1]
Lấy: (1, 'xanh', 1) | ans: [0, 1, -1]
  thêm: (2, 'đỏ', 2)
Lấy: (2, 'đỏ', 2) | ans: [0, 1, 2]


## 12. Code đúng format để nộp LeetCode

Dùng một class `Solution`, tên hàm `shortestAlternatingPaths`, đủ 3 tham số và trả `List[int]`. LeetCode sẽ tự tạo dữ liệu đầu vào và gọi method; không cần gọi `print` trong bản nộp.

In [9]:
from typing import List

class Solution:
    def shortestAlternatingPaths(
        self, n: int, redEdges: List[List[int]], blueEdges: List[List[int]]
    ) -> List[int]:
        from collections import deque
        graph = [[[] for _ in range(n)] for _ in range(2)]
        for u, v in redEdges:
            graph[0][u].append(v)
        for u, v in blueEdges:
            graph[1][u].append(v)
        answer = [-1]*n
        queue = deque([(0,-1,0)])
        seen = {(0,-1)}
        while queue:
            u, last_color, distance = queue.popleft()
            if answer[u] == -1:
                answer[u] = distance
            for color in (0,1):
                if color == last_color:
                    continue
                for v in graph[color][u]:
                    if (v,color) not in seen:
                        seen.add((v,color))
                        queue.append((v,color,distance+1))
        return answer

assert Solution().shortestAlternatingPaths(3,[[0,1],[1,2]],[]) == [0,1,-1]
assert Solution().shortestAlternatingPaths(3,[[0,1]],[[2,1]]) == [0,1,-1]
print('LeetCode: ví dụ 1, 2 PASS')

LeetCode: ví dụ 1, 2 PASS


## 13. Vì sao BFS đúng? (invariant)

Trong hàng đợi, trạng thái được lấy ra theo **độ dài đường đi không giảm**, vì mỗi bước mới luôn dài hơn đúng 1 cạnh.

- Khi một trạng thái `(u,last_color)` được **đưa vào queue lần đầu**, độ dài đã là ngắn nhất đến đúng trạng thái ấy.
- Mọi cạnh hợp lệ từ trạng thái ấy đều được xét: chỉ chặn cạnh **cùng màu** với cạnh trước.
- Chỉ số đỉnh không đủ biểu diễn trạng thái, nên visited phải xét thêm màu.
- `answer[u]` lấy độ dài nhỏ nhất trong các trạng thái đến u; BFS gặp độ dài nhỏ nhất trước.

Không bỏ sót đường đi tối ưu, cũng không chấp nhận hai cạnh liên tiếp cùng màu.

## 14. Time / Space Complexity

Đặt `n` là số đỉnh, `R=len(redEdges)`, `B=len(blueEdges)`, `m=R+B`.

- Dựng đồ thị: `O(n+m)` thời gian/bộ nhớ.
- Có nhiều nhất **2n+1** trạng thái `(node,last_color)` được duyệt; mỗi trạng thái xét danh sách kề của màu ngược lại. Mỗi cạnh của từng màu được quét tối đa một số lần hằng số.
- **Tổng thời gian: `O(n+m)`; bộ nhớ: `O(n+m)`**.

Đây là lý do BFS tốt hơn thử mọi đường đi khi đồ thị có chu trình: ta không lặp vô hạn.

## 15. Oracle độc lập: Relaxation theo số bước (chậm hơn)

Để kiểm thử BFS, ta cần một phương án khác **không dùng queue**. Gọi `dist[u][color]` là số cạnh ngắn nhất từ 0 tới u với **cạnh cuối** có màu `color`. Ban đầu đặt `dist[0][đỏ]=dist[0][xanh]=0` — cách diễn đạt rằng cạnh đầu có thể mang một trong hai màu.

Mỗi vòng, lấy **bản sao dist cũ**, cho phép thêm đúng **một cạnh** và cập nhật bản mới. Vì có `2n` trạng thái và mỗi cạnh dài 1, một đường đi ngắn nhất không cần lặp một trạng thái; tối đa `2n-1` bước (lặp `2n` lần là đủ).

Độ phức tạp xấp xỉ `O(n(n+m))`. Chỉ dùng làm oracle trên dữ liệu nhỏ, không phải bản nộp.

In [10]:
def shortest_alternating_relaxation(n, red_edges, blue_edges):
    INF = 10**9
    dist = [[INF, INF] for _ in range(n)]
    dist[0] = [0,0]
    for _ in range(2*n):
        old = [row[:] for row in dist]
        for u,v in red_edges:
            dist[v][RED] = min(dist[v][RED], old[u][BLUE]+1)
        for u,v in blue_edges:
            dist[v][BLUE] = min(dist[v][BLUE], old[u][RED]+1)
        if dist == old:
            break
    return [(-1 if min(row)==INF else min(row)) for row in dist]

assert shortest_alternating_relaxation(3,red_demo,blue_demo) == [0,1,2]
assert shortest_alternating_relaxation(3,[[0,1],[1,2]],[]) == [0,1,-1]
print('Oracle relaxation: PASS')

Oracle relaxation: PASS


## 16. Bộ test bắt buộc — biên, vòng lặp và hai màu đến cùng đỉnh

Những ca dễ sai nhất:

- **n=1**: không có cạnh, vẫn trả `[0]`.
- Hai cạnh **đỏ→đỏ**: không hợp lệ.
- Có cả cạnh đỏ và xanh từ 0 tới 1: phải giữ **hai trạng thái khác nhau**.
- **Self-loop**, **parallel edge**: không được thăm mãi; trả khoảng cách ngắn nhất.
- Đồ thị không có đường tới một số đỉnh ⇒ `-1`.

In [11]:
tests = [
    (1, [], [], [0]),
    (1, [[0,0]], [[0,0]], [0]),
    (3, [[0,1],[1,2]], [], [0,1,-1]),
    (3, [[0,1]], [[2,1]], [0,1,-1]),
    (3, [[0,1]], [[1,2]], [0,1,2]),
    (3, [[0,1],[1,2]], [[0,1]], [0,1,2]),
    (4, [[0,1],[2,3]], [[1,2]], [0,1,2,3]),
    (3, [[0,1],[0,1]], [[1,1],[1,2]], [0,1,2]),
    (3, [], [], [0,-1,-1]),
]
for n,r,b,expected in tests:
    assert Solution().shortestAlternatingPaths(n,r,b) == expected, (n,r,b)
    assert shortest_alternating_bfs(n,r,b) == expected
    assert shortest_alternating_relaxation(n,r,b) == expected
print('Các ca mẫu/biên:',len(tests),'PASS')

Các ca mẫu/biên: 9 PASS


## 17. Kiểm thử đối chiếu ngẫu nhiên

Sinh các đồ thị nhỏ có hướng, cho phép cạnh tự khép và cạnh trùng. So 3 cách: BFS chính, code LeetCode và oracle relaxation. Khi cả ba trùng nhau trên hàng trăm đồ thị, nguy cơ lỗi logic giảm đáng kể (dù không thay cho chứng minh).

In [12]:
import random
rng = random.Random(1129)
for _ in range(250):
    n = rng.randint(1,8)
    red_edges = [[rng.randrange(n),rng.randrange(n)] for _ in range(rng.randint(0,14))]
    blue_edges = [[rng.randrange(n),rng.randrange(n)] for _ in range(rng.randint(0,14))]
    ref = shortest_alternating_relaxation(n,red_edges,blue_edges)
    assert shortest_alternating_bfs(n,red_edges,blue_edges) == ref
    assert Solution().shortestAlternatingPaths(n,red_edges,blue_edges) == ref
print('250 đồ thị ngẫu nhiên: PASS')
n=100
dense_red = [[u,(u*11+k)%n] for u in range(n) for k in range(4)]
dense_blue = [[u,(u*7+k)%n] for u in range(n) for k in range(4)]
assert len(Solution().shortestAlternatingPaths(n,dense_red,dense_blue)) == n
print('n=100, 400 cạnh mỗi màu: PASS')

250 đồ thị ngẫu nhiên: PASS
n=100, 400 cạnh mỗi màu: PASS


## 18. Lỗi thường gặp: sai → vì sao → sửa

1. **Chỉ dùng `visited_nodes={u}`:** đánh mất trường hợp đến cùng u bằng màu khác. Sửa thành `seen={(u,last_color)}`.
2. **Đảo chiều cạnh:** thêm `g[v].append(u)` thay vì `g[u].append(v)`. Đồ thị có hướng.
3. **Chỉ bắt đầu với đỏ:** mất các đường phải bắt đầu bằng xanh. Sửa bằng `last_color=-1` hoặc khởi tạo hai trạng thái màu giả.
4. **Cho phép màu trùng:** vô tình chấp nhận đỏ→đỏ. Sửa bằng `if color==last_color: continue`.
5. **Đánh dấu visited khi pop:** vẫn có thể đúng nhưng đẩy trùng nhiều lần, tốn tài nguyên. Nên đánh dấu khi **append queue**.

In [13]:
red_same = [[0,1],[1,2]]
assert shortest_paths_ignore_colors(3,red_same,[]) == [0,1,2]  # sai nếu nộp
assert Solution().shortestAlternatingPaths(3,red_same,[]) == [0,1,-1]  # đúng
print('Phản ví dụ BFS bỏ màu: PASS')

Phản ví dụ BFS bỏ màu: PASS


## 19. Mẹo nhận diện bài toán “trạng thái mở rộng”

Dấu hiệu: **cùng một vị trí** nhưng khả năng đi tiếp **khác nhau vì lịch sử ngắn** (màu trước, số chìa khóa, số lần phá tường, hướng trước...). Hãy thay `visited[node]` bằng `visited[(node, trạng_thái_bổ_sung)]`.

Tuy nhiên, chỉ thêm **đúng thông tin ảnh hưởng đến lựa chọn tương lai**; không mang toàn bộ đường đi nếu không cần. Ở đây chỉ cần màu cạnh cuối, không cần chuỗi mọi màu đã đi.

In [14]:
# Biến thể: trả về số đỉnh có thể tới bằng đường màu xen kẽ.
def count_reachable(n, red_edges, blue_edges):
    ans = shortest_alternating_bfs(n,red_edges,blue_edges)
    return sum(d >= 0 for d in ans)

assert count_reachable(3, [[0,1],[1,2]], []) == 2
assert count_reachable(3, [[0,1]], [[1,2]]) == 3
print('Biến thể đếm đỉnh đến được: PASS')

Biến thể đếm đỉnh đến được: PASS


## 20. Năm bài tập luyện tập (dễ → khó)

**Bài 1 — Tính tay:** Với đỏ `0→1`, xanh `1→2`, đỏ `2→3`, viết các trạng thái BFS và mảng answer.

**Bài 2 — Đoán output:** Với đỏ `0→1→2`, xanh rỗng, điều gì xảy ra ở đỉnh 2? Vì sao?

**Bài 3 — TODO cấu trúc dữ liệu:** Viết `exercise_build_graph(n, red, blue)` trả hai adjacency lists theo màu.

**Bài 4 — TODO thuật toán:** Viết `exercise_shortest_alternating_paths` độc lập, dùng queue và visited theo `(node,color)`.

**Bài 5 — Biến thể:** Viết `exercise_count_unreachable` đếm số đỉnh mà không có đường màu xen kẽ từ 0.

Các test sau đây được **comment** để người học chạy khi hoàn thiện TODO, không làm lỗi nút Run All.

In [15]:
def exercise_build_graph(n, red_edges, blue_edges):
    # TODO: tạo g[2][n] rồi append cạnh đúng màu, đúng hướng
    pass

def exercise_shortest_alternating_paths(n, red_edges, blue_edges):
    # TODO: queue khởi tạo (0,-1,0), seen lưu tuple (node,last_color)
    pass

def exercise_count_unreachable(n, red_edges, blue_edges):
    # TODO: xét kết quả đường đi có bằng -1 không
    pass

# Sau khi hoàn thiện, bỏ comment:
# assert exercise_build_graph(3,[[0,1]],[[1,2]]) == [[[1],[],[]], [[],[2],[]]]
# assert exercise_shortest_alternating_paths(3,[[0,1]],[[1,2]]) == [0,1,2]
# assert exercise_count_unreachable(3,[[0,1],[1,2]],[]) == 1

## 21. Gợi ý theo 3 mức

| Bài | Gợi ý 1: ý tưởng | Gợi ý 2: cú pháp | Gợi ý 3: sát code |
|---|---|---|---|
| 1 | Từng cạnh phải đổi màu | Độ dài tăng 1 | `0, (1,RED), (2,BLUE), (3,RED)` |
| 2 | Không có cạnh xanh | Không được đi đỏ→đỏ | Chỉ có đỉnh 0 và 1 đạt được |
| 3 | Hai nhóm kề | `[[] for _ in range(n)]` | `g[0][u].append(v)` |
| 4 | BFS trạng thái | `deque`, `set` chứa tuple | `if color != last` rồi thêm `(v,color)` |
| 5 | Đếm `-1` | `sum(cond for...)` | `sum(d==-1 for d in dist)` |

## 22. Đáp án tham khảo (chỉ đọc sau khi tự làm)

**Bài 1:** đáp án `[0,1,2,3]`; các trạng thái `(0,-1)`, `(1,đỏ)`, `(2,xanh)`, `(3,đỏ)`; mỗi cạnh đổi màu.

**Bài 2:** `[0,1,-1]`; sau khi đi 0→1 bằng đỏ, không được đi thêm cạnh đỏ 1→2.

Bài 3–5 có lời giải dưới đây, dùng tên hàm khác để **không ghi đè code TODO** của bạn.

In [16]:
def answer_build_graph(n, red_edges, blue_edges):
    g = [[[] for _ in range(n)] for _ in range(2)]
    for u,v in red_edges:g[0][u].append(v)
    for u,v in blue_edges:g[1][u].append(v)
    return g

def answer_shortest_alternating_paths(n, red_edges, blue_edges):
    graph = answer_build_graph(n,red_edges,blue_edges)
    answer = [-1]*n
    queue = deque([(0,-1,0)])
    seen = {(0,-1)}
    while queue:
        u,last,d = queue.popleft()
        if answer[u] == -1:answer[u] = d
        for color in (0,1):
            if color == last:continue
            for v in graph[color][u]:
                if (v,color) not in seen:
                    seen.add((v,color))
                    queue.append((v,color,d+1))
    return answer

def answer_count_unreachable(n, red_edges, blue_edges):
    return sum(d == -1 for d in answer_shortest_alternating_paths(n,red_edges,blue_edges))

assert answer_build_graph(3,[[0,1]],[[1,2]]) == [[[1],[],[]], [[],[2],[]]]
assert answer_shortest_alternating_paths(3,[[0,1]],[[1,2]]) == [0,1,2]
assert answer_count_unreachable(3,[[0,1],[1,2]],[]) == 1
print('Đáp án tham khảo 3–5: PASS')

Đáp án tham khảo 3–5: PASS


## 23. Template ghi nhớ và checklist submit

```text
Đồ thị có ràng buộc lịch sử ngắn:
state = (node, thông_tin_quyết_định_bước_sau)
queue BFS duyệt state theo số cạnh tăng dần
visited lưu trọn state, không chỉ node
```

- [ ] Tôi nhớ cạnh **có hướng**.
- [ ] Tôi biết tại sao cạnh đầu có **2 lựa chọn màu**.
- [ ] Tôi không cho đi hai cạnh cùng màu liên tiếp.
- [ ] Tôi lưu `seen` theo `(u,last_color)`.
- [ ] Tôi khởi tạo `answer[0]=0` qua lần pop đầu tiên.
- [ ] Đỉnh không đến được vẫn giữ `-1`.
- [ ] Tôi tự kiểm thử self-loop, cạnh song song, chu trình, hai màu cùng đầu-cuối.